In [ ]:
import os
import sys
import zipfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules


def find_repo_root(start: str) -> Path:
    for root, _, files in os.walk(start):
        if "pyproject.toml" in files and "requirements.txt" in files:
            return Path(root)
    raise FileNotFoundError("Could not find repo root with pyproject.toml")


repo_root = None
if IN_COLAB:
    try:
        repo_root = find_repo_root("/content")
    except FileNotFoundError:
        from google.colab import files

        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("Please upload the repo zip to continue.")
        zip_name = next(iter(uploaded))
        with zipfile.ZipFile(zip_name, "r") as zip_ref:
            zip_ref.extractall("/content")
        repo_root = find_repo_root("/content")
else:
    repo_root = find_repo_root(os.getcwd())

os.chdir(repo_root)
print("Repo root:", repo_root)


Saving TP.zip to TP.zip


In [ ]:
!pip install -q -r requirements.txt
!pip install -q -e .


In [ ]:
import random
import numpy as np

SEED = 42
SYNTHETIC_V1_SEED = 0
SYNTHETIC_V2_SEED = 2
ALNS_SEED = 1

random.seed(SEED)
np.random.seed(SEED)

TRAINING_DIR = (
    Path(repo_root)
    / "bin_packing_optimization"
    / "hybrid_learning_metaheuristics"
    / "hybrid_alns"
    / "repair_model_training"
)
DATA_DIR = TRAINING_DIR / "training_data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

os.chdir(TRAINING_DIR)
print("Training dir:", TRAINING_DIR)

from bin_packing_optimization.hybrid_learning_metaheuristics.hybrid_alns.repair_model_training.generate_dataset import (
    GenerateDatasetConfig,
    generate_dataset,
)
from bin_packing_optimization.hybrid_learning_metaheuristics.hybrid_alns.repair_model_training.collect_alns_states import (
    CollectAlnsStatesConfig,
    collect_alns_states,
)
from bin_packing_optimization.hybrid_learning_metaheuristics.hybrid_alns.repair_model_training.train_repair_model import (
    TrainRepairModelConfig,
    train_repair_model,
)


[Errno 2] No such file or directory: 'content/TP/TP/5_hybrid_ml_metaheuristics/hybrid_alns'
/content/TP/TP/5_hybrid_ml_metaheuristics/hybrid_alns
PHASE 1 : GÉNÉRATION DU DATASET
Generating 4000 synthetic instances...
  Taille items: [50, 200]
  Max négatifs par positif: 5
  Workers: 1
Generating instances: 100% 4000/4000 [00:21<00:00, 188.32inst/s]
✅ Dataset généré: 1313891 rows × 11 features
   Positive rate: 0.2194 (attendu min: 0.1667)
   Classe 0 (négatifs): 1,025,570 samples
   Classe 1 (positifs):  288,321 samples

PHASE 2 : SPLIT TRAIN/TEST
✅ Training set:  1,116,807 samples
✅ Test set:      197,084 samples

PHASE 3 : ENTRAÎNEMENT DU MODÈLE
⚡ Fast mode activé: entraînement allégé pour debug/itérations rapides.
Training GradientBoosting avec hyperparameters optimisés:
  n_estimators      : 180
  max_depth         : 4
  learning_rate     : 0.05
  subsample         : 0.8
  min_samples_split : 20
  min_samples_leaf  : 10
  early_stopping    : OUI (validation_fraction=0.1)

Suivi d'a

In [ ]:
generate_dataset(
    GenerateDatasetConfig(
        instances=4000,
        n_min=50,
        n_max=200,
        max_negatives=5,
        seed=SYNTHETIC_V1_SEED,
        workers=1,
        output=str(DATA_DIR / "synthetic_v1.pkl"),
    )
)


[Errno 2] No such file or directory: 'content/TP/TP/5_hybrid_ml_metaheuristics/hybrid_alns'
/content/TP/TP/5_hybrid_ml_metaheuristics/hybrid_alns
Running ALNS on 500 instances to collect repair states...
Collected 309539 rows  (pos_rate=0.297)
Saved to: alns_states_v1.pkl
Next step: retrain with --augment-with alns_states_v1.pkl


In [ ]:
train_repair_model(
    TrainRepairModelConfig(
        data=[str(DATA_DIR / "synthetic_v1.pkl")],
        output=str(TRAINING_DIR / "repair_model_v1.pkl"),
        seed=SEED,
        min_roc_auc=0.80,
        min_average_precision=0.60,
        require_alns_states=False,
        cv_folds=5,
        no_learning_curves=True,
        no_plots=True,
    )
)


[Errno 2] No such file or directory: 'content/TP/TP/5_hybrid_ml_metaheuristics/hybrid_alns'
/content/TP/TP/5_hybrid_ml_metaheuristics/hybrid_alns
PHASE 1 : GÉNÉRATION DU DATASET
Generating 2000 synthetic instances...
  Taille items: [50, 200]
  Max négatifs par positif: 3
  Workers: 2
Generating instances: 100% 2000/2000 [00:11<00:00, 167.37inst/s]
✅ Augmenté avec 309539 ALNS states → total: 802249 rows
✅ Dataset généré: 492710 rows × 11 features
   Positive rate: 0.2932 (attendu min: 0.2500)
   Classe 0 (négatifs): 565,906 samples
   Classe 1 (positifs):  236,343 samples

PHASE 2 : SPLIT TRAIN/TEST
✅ Training set:  681,911 samples
✅ Test set:      120,338 samples

PHASE 3 : ENTRAÎNEMENT DU MODÈLE
Training GradientBoosting avec hyperparameters optimisés:
  n_estimators      : 500
  max_depth         : 6
  learning_rate     : 0.03
  subsample         : 0.75
  min_samples_split : 20
  min_samples_leaf  : 10
  early_stopping    : OUI (validation_fraction=0.1)

Suivi d'avancement de l'entr

In [ ]:
collect_alns_states(
    CollectAlnsStatesConfig(
        model_path=str(TRAINING_DIR / "repair_model_v1.pkl"),
        instances=500,
        n_min=50,
        n_max=200,
        max_negatives=5,
        iterations=200,
        seed=ALNS_SEED,
        output=str(DATA_DIR / "alns_states_v1.pkl"),
    )
)


/content/hybrid_alns/hybrid_alns/repair_model_training
PHASE 1: DATASET GENERATION
Generating 4000 synthetic instances...
  Item size range: [50, 200]
  Maximum negatives per positive: 3
  Workers: 4
Generating dataset: 100% 4000/4000 [00:08<00:00, 467.10it/s]
Dataset generated: 566381 rows x 11 features
   Positive rate: 0.4036 (expected min: 0.2500)
   Class 0 (negatives): 337,772 samples
   Class 1 (positives):  228,609 samples

PHASE 2: TRAIN/TEST SPLIT
Training set:  481,423 samples
Test set:      84,958 samples

PHASE 3: MODEL TRAINING
Training GradientBoosting with tuned hyperparameters:
  n_estimators      : 500
  max_depth         : 6
  learning_rate     : 0.03
  subsample         : 0.75
  min_samples_split : 20
  min_samples_leaf  : 10
  early_stopping    : YES (validation_fraction=0.1)
      Iter       Train Loss      OOB Improve   Remaining Time 
         1           1.3624           0.0235           42.43m
         2           1.3400           0.0227           40.04m
     

In [ ]:
generate_dataset(
    GenerateDatasetConfig(
        instances=2000,
        n_min=50,
        n_max=200,
        max_negatives=3,
        seed=SYNTHETIC_V2_SEED,
        workers=1,
        output=str(DATA_DIR / "synthetic_v2.pkl"),
    )
)

train_repair_model(
    TrainRepairModelConfig(
        data=[
            str(DATA_DIR / "synthetic_v2.pkl"),
            str(DATA_DIR / "alns_states_v1.pkl"),
        ],
        output=str(TRAINING_DIR / "repair_model_v2.pkl"),
        seed=SEED,
        min_roc_auc=0.80,
        min_average_precision=0.60,
        cv_folds=3,
        no_learning_curves=True,
        no_plots=True,
    )
)

RUN_BENCHMARK = False
if RUN_BENCHMARK:
    from bin_packing_optimization.hybrid_learning_metaheuristics.hybrid_alns import (
        hybrid_alns_solver,
    )
    from bin_packing_optimization.utilities.benchmarking import create_benchmark

    benchmark = create_benchmark(
        dataset_key="falkenauer-u",
        solver_module=hybrid_alns_solver,
        time_limit=None,
    )
    benchmark.run(method=None, method_args={"max_iterations": 500})
    csv_path = benchmark.save_results_to_csv()
    print(csv_path)


[Errno 2] No such file or directory: '/content/TP/TP'
/content/hybrid_alns/hybrid_alns/repair_model_training
/usr/bin/python3: Error while finding module specification for 'bin_packing_optimization.hybrid_learning_metaheuristics.hybrid_alns.repair_model_training.train_full_pipeline' (ModuleNotFoundError: No module named 'bin_packing_optimization')
